# Method 5 — Graph Total Variation (TV), solved with ADMM

**Idea in one paragraph.** The observation is `y = x + noise + a few outliers`. We assume the true signal is **constant on regions** of the graph, with a few sharp jumps between regions. We look for the `x_hat` that stays close to `y` while making neighbouring nodes **equal** wherever possible. Disagreement between neighbours is measured with the absolute value `|x_i − x_j|` (Total Variation), not the square `(x_i − x_j)²`. The absolute value tolerates a few large jumps and sets most differences to exactly 0, so whole regions share one value. The problem has no closed form, so we solve it with **ADMM**, an iterative method whose every step is a simple formula.

Notebook map:
0. Vocabulary (TV, incidence matrix, soft-thresholding, ADMM, residuals, ...).
1. Load data (Mục III/IV of the assignment), the shared code block from the group doc, evaluation helpers.
2. The method: assumptions, objective, two properties of the solution, the ADMM algorithm, code, the function for the group (`solve_tv`), correctness checks and convergence.
3. Final `x_hat` on the real data (λ chosen by `cv_score`), save `results/method_5/xhat_5.npy`, figure in `figures/method_5/`.
4. Result row for the group comparison table (`results/method_5/result_5.json`).
5. Write-up (Mục VI, 4 parts).

Appendices (optional reading): A. parameter choice with `cv_score`; B. synthetic data (one seed, seeds 0-19, blind protocol, numbers behind the limitations); C. using method 6's outlier flags with TV; D. notes for the group meeting.

> **Note:** the MSSV list in Section 1 is still the placeholder list. Once the group fixes the real list, edit it and re-run the whole notebook: λ, every number, figure and `xhat_5.npy` are recomputed. The text avoids hard-coded results and points to the printed outputs instead.

## Vocabulary (read this first)

| term | short meaning |
|---|---|
| node `i`, neighbour `j`, edge weight `w_ij` | nodes 0 … n−1; `j` joined to `i` by an edge; bigger `w_ij` = the two sensors are closer and should agree more |
| `y` / `x` / `x_hat` | observed noisy values / true unknown values / our estimate of `x` |
| piecewise constant | a signal that takes one value on each region of the graph and jumps between regions |
| **Total Variation (TV)** | $\mathrm{TV}(x)=\sum_{(i,j)\in E} w_{ij}\,\lvert x_i-x_j\rvert$: the weighted sum of the absolute gaps across edges. Small = neighbours agree. Compare with the roughness `xᵀLx = Σ w_ij (x_i − x_j)²` used by methods 1, 3, 4, 6, 7 (example below) |
| `λ` (`lam`) | how much TV matters compared with staying close to `y`. Model parameter, chosen with `cv_score` |
| incidence matrix `D` | one row per edge `e = (i, j)`: +1 in column `i`, −1 in column `j`. So `(Dx)_e = x_i − x_j` and $\mathrm{TV}(x)=\sum_e w_e\,\lvert (Dx)_e\rvert$ |
| weighted degree `d_i` | `d_i = Σ_j w_ij`, the total weight of node `i`'s edges |
| subgradient | the "slope" of $\lvert t\rvert$ where it has a corner: −1 for `t < 0`, +1 for `t > 0`, any number in [−1, 1] at `t = 0` |
| soft-thresholding `S_τ(v)` | shrink `v` towards 0 by `τ`, and set it to exactly 0 if $\lvert v\rvert\le\tau$. Example with `τ = 1`: −3 → −2, −0.5 → 0, 0.5 → 0, 3 → 2. It solves $\min_z\ \tau\lvert z\rvert+\tfrac12(z-v)^2$ |
| prox (proximal operator) | the general version: `prox_f(v) = argmin_z f(z) + ½‖z − v‖²`, "move towards making `f` small, but stay near `v`" |
| ADMM | Alternating Direction Method of Multipliers: split a hard problem into easy pieces, solve them in turn, and use a running correction (`u`) until the pieces agree |
| primal / dual residual | the two progress measures of ADMM: how much the pieces still disagree / how much they still move. Both must become small to stop |
| `ρ` (`rho`) | ADMM step parameter. Changes the speed only, never the answer |
| Huber loss | `½r²` for $\lvert r\rvert\le\delta$, linear beyond: big residuals pull with a bounded force |
| weighted median | sort the neighbours' values, walk down adding weights, stop at half of the total weight |
| `mask` | True = node observed; False = node hidden (used by `cv_score`) |
| `cv_score` | hide 1/5 of the nodes, predict them from the rest, median absolute error (lower is better). Method 6, Appendix A explains it in detail |

### Details: TV versus roughness on a chain

Chain `0 — 1 — 2 — 3 — 4`, all weights 1.

| signal | values | gaps along edges | roughness `xᵀLx` (squares) | TV (absolute values) |
|---|---|---|---|---|
| smooth ramp | 10, 11, 12, 13, 14 | 1, 1, 1, 1 | 4 | 4 |
| one jump | 10, 10, 10, 14, 14 | 0, 0, 4, 0 | **16** | **4** |
| spike | 10, 10, 30, 10, 10 | 0, 20, 20, 0 | 800 | 40 |

The ramp and the jump have the **same TV** (4): TV does not care whether the change from 10 to 14 happens in small steps or in one jump. Roughness charges the jump 4 times more (16 vs 4), so a method based on `xᵀLx` prefers to spread any jump over many edges, which blurs region boundaries. TV keeps the jump sharp. Both penalise the spike heavily, so both pull an isolated outlier back towards its neighbours.

### Details: why TV makes regions flat

The soft-threshold step of ADMM (Section 2.4) handles each edge on its own: a gap smaller than the threshold `λ w_ij / ρ` is set to **exactly 0**. Gaps that are only noise disappear and the two nodes get the same value; only gaps large enough to be real jumps survive. This is the same mechanism that makes the Lasso set coefficients to exactly 0.

## 1. Data and shared code

In [ ]:
# ---- Mục III/IV of the assignment: group list, load G, y, positions ---------
# REPLACE the placeholder list below with ALL real MSSV of the group (strings, keep leading zeros).
NHOM_MSSV = [
    "20260001",
    "20260002",
    "20260003",
    "20260004",
    "20260005",
    "20260006",
    "20260007",
]

# Checks (do not edit).
NHOM_MSSV = [str(s).strip() for s in NHOM_MSSV]
assert NHOM_MSSV and all(NHOM_MSSV), "Missing MSSV in the group list"
assert len(NHOM_MSSV) == len(set(NHOM_MSSV)), "Duplicate MSSV in the group list"

# LAST LINE: the representative MSSV that selects the group's data.
MSSV_DAI_DIEN = NHOM_MSSV[0]

import json, hashlib
from pathlib import Path
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Repository root = the folder that holds public_cases.json (works from any subfolder).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "public_cases.json").exists())


def assign_mssv_to_case(student_id, cases):
    """Map any MSSV deterministically to an EXISTING case ID in public_cases.json (unchanged assignment code)."""
    sid = str(student_id).strip()
    if not sid:
        raise ValueError("MSSV must not be empty")
    if not cases:
        raise ValueError("No case in public_cases.json")
    if sid in cases:
        return sid
    case_ids = sorted(cases, key=lambda x: (0, int(x)) if x.isdecimal() else (1, x))
    k = len(case_ids)
    if sid.isdecimal():
        index = int(sid) % k
    else:
        index = int.from_bytes(hashlib.sha256(sid.encode("utf-8")).digest(), "big") % k
    return case_ids[index]


def get_problem(student_id, file_path=ROOT / "public_cases.json"):
    sid = str(student_id).strip()
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError("public_cases.json not found at the repository root.")
    db = json.loads(path.read_text(encoding="utf-8"))
    assigned_id = assign_mssv_to_case(sid, db["cases"])
    case = db["cases"][assigned_id]
    G = nx.Graph()
    for i, xy in enumerate(case["positions"]):
        G.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in case["edges"]:
        G.add_edge(int(u), int(v), weight=float(w))
    y = np.asarray(case["y"], dtype=float)
    if not nx.is_connected(G) or len(y) != G.number_of_nodes():
        raise ValueError("Invalid graph data")
    return G, y


G, y = get_problem(MSSV_DAI_DIEN)
positions = nx.get_node_attributes(G, "pos")
with open(ROOT / "public_cases.json", encoding="utf-8") as _f:
    _case_db = json.load(_f)["cases"]
print("Representative MSSV:", MSSV_DAI_DIEN)
print("Assigned case ID:", assign_mssv_to_case(MSSV_DAI_DIEN, _case_db))
print("Group:", ", ".join(NHOM_MSSV))
print(f"Nodes = {G.number_of_nodes()} | Edges = {G.number_of_edges()}")
print("y =", np.round(y, 4))

In [ ]:
# ===== Shared "contract" block from the group doc — keep unchanged =====
n = len(y)
L = nx.laplacian_matrix(G, nodelist=range(n), weight="weight").toarray()

# (a) Synthetic data with a KNOWN x, used to measure RMSE
def make_synthetic(G, y, seed):
    rng = np.random.default_rng(seed)
    lam, U = np.linalg.eigh(L)
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam[:5]))
    x = np.median(y) + np.std(y) * x / np.std(x)
    sigma = 0.1 * np.std(y)
    y_syn = x + rng.normal(0, sigma, n)
    idx = rng.choice(n, 3, replace=False)                      # 3 outliers
    y_syn[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, y_syn

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))

# (b) Parameter choice on the real data: hide 1/5 of the nodes, predict them back
def cv_score(solve, y, params, k=5, seed=0):
    folds = np.random.default_rng(seed).permutation(n) % k
    err = []
    for f in range(k):
        mask = folds != f                                      # True = observed
        x = solve(G, y, mask=mask, **params)
        err.append(np.abs(x[~mask] - y[~mask]))
    return np.median(np.concatenate(err))                      # median so outliers do not dominate

In [ ]:
# ---- Libraries, output folders, plotting style, evaluation helpers -----------
import time
from matplotlib.colors import LinearSegmentedColormap, Normalize
from scipy.linalg import cho_factor, cho_solve

FIG_DIR = ROOT / "figures" / "method_5"; FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR = ROOT / "results" / "method_5"; RES_DIR.mkdir(parents=True, exist_ok=True)
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # fixed colour order
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 12,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK, "font.size": 10,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "figure.dpi": 110,
})

SEEDS = range(20)                                              # synthetic seeds agreed in the group doc
SYN = [make_synthetic(G, y, s) for s in SEEDS]                 # (x_true, y_syn) pairs, built once
rmse_y = np.array([rmse(ys, xt) for xt, ys in SYN])            # baseline: return y unchanged

def synth_rmse(params, data=SYN):
    # RMSE against the KNOWN x on synthetic data, one value per seed.
    return np.array([rmse(solve_tv(G, ys, **params), xt) for xt, ys in data])

def cv_mean(solve, y, params, seeds=range(5)):
    # cv_score averaged over 5 random splits (what method 6 reports), so one lucky split does not decide.
    return float(np.mean([cv_score(solve, y, params, seed=sd) for sd in seeds]))

## 2. The method

### 2.1 Assumptions (Mục VI, part 1)

- **Observations.** $y_i = x_i + \varepsilon_i$. Most $\varepsilon_i$ are small, independent measurement noise; a few nodes have large errors (outliers).
- **The signal is constant on regions.** Two nodes joined by a heavy edge usually measure the same value, but the graph may contain a few **jumps**: boundaries between regions of the sensor network. Equivalently, the vector of edge differences $\{x_i-x_j\}_{(i,j)\in E}$ is **sparse**: most entries are 0, a few are large.
- **Weights are trust.** $w_{ij}$ says how strongly we believe nodes $i$ and $j$ measure the same thing; a heavy edge costs more to "cut" with a jump.

### 2.2 Objective (Mục VI, part 2)

$$\hat x=\arg\min_x\ \underbrace{\sum_{i\in\mathcal O}\ell(x_i-y_i)}_{\text{stay close to the data}}\;+\;\lambda\underbrace{\sum_{(i,j)\in E}w_{ij}\,|x_i-x_j|}_{\text{TV: agree with neighbours}},\qquad \ell(r)=\tfrac12r^2\ \text{or}\ \mathrm{Huber}_\delta(r),$$

where $\mathcal O$ is the set of observed nodes: all 32 nodes on the real problem, the non-hidden nodes inside `cv_score`.

- **Why $|x_i-x_j|$ and not $(x_i-x_j)^2$:** see "TV versus roughness" in the Vocabulary. A few large jumps are affordable, and many differences become exactly 0.
- **Why Huber as an option.** With squared loss an outlier pulls on $\hat x$ in proportion to its size. Huber is quadratic for small residuals and linear beyond $\delta$, so a large outlier pulls with a bounded force. Whether this helps is decided by the data (Appendix A).
- **Convex.** Both terms are convex, so every local minimum is the global minimum and ADMM converges to it.
- **Limits.** $\lambda\to0$ gives $\hat x=y$ (no smoothing). $\lambda\to\infty$ forces every difference to 0, so $\hat x$ is a constant; with squared loss that constant is $\mathrm{mean}(y)$. Section 2.6 checks both.
- **Two kinds of parameter.** $\lambda$ (and $\delta$) are **model** parameters: they change the answer and are chosen by `cv_score`. $\rho$ is an **algorithm** parameter: it only changes how fast we reach the same answer.

**Small example (one node, squared loss).** Node $i$ reads $y_i = 14$ and has two neighbours fixed at 10 (weights 1). Minimise $\tfrac12(x-14)^2+\lambda(|x-10|+|x-10|)$ for $x>10$: the derivative is $(x-14)+2\lambda=0$, so $x = 14-2\lambda$. With $\lambda=0.5$: $x=13$; with $\lambda=1$: $x=12$; with $\lambda\ge2$ the formula would go below 10, so $x$ stops at **exactly 10** — the node joins its neighbours' region. The correction is $14-x=2\lambda=\lambda d_i$ until it joins: this is Property 1 below.

### 2.3 Two properties of the solution

Both follow from the optimality condition (0 belongs to the subgradient of the objective), and both are used to read the results.

1. **Bounded correction.** At an observed node with squared loss, $y_i-\hat x_i=\lambda\sum_j w_{ij}g_{ij}$, where each $g_{ij}\in[-1,1]$ is a subgradient of $|\hat x_i-\hat x_j|$. Hence
$$|y_i-\hat x_i|\le\lambda d_i,\qquad d_i=\textstyle\sum_j w_{ij}.$$
   TV moves each node by at most $\lambda d_i$. Good: one outlier can only distort the result so much. Bad: an outlier larger than its cap is only partly corrected. (With Huber the same bound holds for the clipped residual $\mathrm{clip}(y_i-\hat x_i,\pm\delta)$.) Checked in Section 3.
2. **A hidden node takes the weighted median of its neighbours.** A hidden node has no data term, so $\hat x_i$ minimises $\sum_j w_{ij}|x_i-\hat x_j|$ — by definition a weighted median of its neighbours' values (the same weighted median method 6 uses in its Stage 1). A prediction at a hidden node therefore ignores one abnormal neighbour, unlike the weighted average of methods 1 and 6. Checked in Section 2.6.

### 2.4 The ADMM algorithm (Mục VI, part 3)

**Why an iterative method.** $|\cdot|$ has no derivative at 0, so we cannot set a gradient to zero and solve one linear system as Tikhonov does. ADMM splits the problem into pieces that each have a closed-form answer and alternates between them.

**Incidence matrix.** $D\in\mathbb R^{m\times n}$; row $e=(i,j)$ has $+1$ in column $i$ and $-1$ in column $j$ (unweighted), so $(Dx)_e=x_i-x_j$, $\;\mathrm{TV}(x)=\sum_e w_e|(Dx)_e|$, and $D^\top D$ is the **unweighted** Laplacian.

**Splitting.** Introduce $z_1=x$ (carries the data term) and $z_2=Dx$ (carries the TV term). The problem becomes
$$\min_{x,z_1,z_2}\ f(z_1)+g(z_2)\quad\text{s.t.}\quad \begin{bmatrix}I\\ D\end{bmatrix}x-\begin{bmatrix}z_1\\ z_2\end{bmatrix}=0,$$
with $f(z_1)=\sum_{i\in\text{obs}}\ell(z_{1,i}-y_i)$ and $g(z_2)=\lambda\sum_e w_e|z_{2,e}|$. Scaled-form ADMM (Boyd et al., 2011, §3.1.1), dual variables $u_1,u_2$:

| Step | Formula | Meaning |
|---|---|---|
| $x$ | $(I+D^\top D)\,x=(z_1-u_1)+D^\top(z_2-u_2)$ | Linear solve; the matrix is **fixed and always invertible** → one Cholesky factorisation |
| $z_1$ | $\operatorname{prox}_{\ell/\rho}(x+u_1)$ on observed nodes; $z_1=x+u_1$ on hidden nodes | Data step, closed form |
| $z_2$ | $S_{\lambda w/\rho}(Dx+u_2)$ | Soft-threshold each edge |
| $u$ | $u_1\mathrel{+}=x-z_1,\ u_2\mathrel{+}=Dx-z_2$ | Dual update |

**Where the four steps come from.** ADMM minimises the augmented Lagrangian $f(z_1)+g(z_2)+\frac\rho2\|x-z_1+u_1\|^2+\frac\rho2\|Dx-z_2+u_2\|^2$ over one block at a time:
- **over $x$:** a sum of two quadratics; setting the gradient to 0 gives $\rho(I+D^\top D)x=\rho\,[(z_1-u_1)+D^\top(z_2-u_2)]$, and $\rho$ cancels;
- **over $z_1$**, node by node: $\min_z \ell(z-y_i)+\frac\rho2(z-v_i)^2$ with $v=x+u_1$. For squared loss the answer is the weighted average $z=(y_i+\rho v_i)/(1+\rho)$. At a hidden node there is no $\ell$, so $z=v_i$: **this is the only place where the mask enters**, which is how hidden values are never used;
- **over $z_2$**, edge by edge: $\min_z \lambda w_e|z|+\frac\rho2(z-v_e)^2$ with $v=Dx+u_2$. The answer is soft-thresholding: shrink $v_e$ towards 0 by $\lambda w_e/\rho$, and set it to **exactly 0** when $|v_e|\le\lambda w_e/\rho$. This is the step that creates the flat regions;
- **$u$** accumulates the remaining disagreement $x-z_1$ and $Dx-z_2$, and pushes the blocks to agree.

Data prox (with $r=v-y_i$): for $\ell(t)=\tfrac12t^2$: $z=y_i+\tfrac{\rho}{1+\rho}r$; for Huber$_\delta$: $z=y_i+\tfrac{\rho}{1+\rho}r$ if $|r|\le\delta(1+1/\rho)$, otherwise $z=y_i+r-\tfrac{\delta}{\rho}\mathrm{sign}(r)$.

**Stop** when the primal residual $r^k=\|[I;D]x-z\|$ and the dual residual $s^k=\rho\|[I\ D^\top](z^k-z^{k-1})\|$ are both below $\varepsilon_{\text{pri}},\varepsilon_{\text{dual}}$ (Boyd et al., 2011, §3.3.1).

**Adaptive $\rho$** (residual balancing, Boyd et al., 2011, §3.4.1): every 10 iterations, double $\rho$ if $r^k>10s^k$ and halve it if $s^k>10r^k$ (rescaling $u$ accordingly). In this splitting the $x$-step matrix is $I+D^\top D$, which **does not contain $\rho$**, so changing $\rho$ needs no new factorisation.

*Why two splits instead of only $z=Dx$?* With $z=Dx$ alone, the $x$-step depends on the mask and has a closed form only for the squared loss. Adding $z_1=x$ allows any loss with a closed-form prox (squared, Huber), and $I+D^\top D$ stays the same across CV folds.

### 2.5 Code

In [ ]:
def incidence_matrix(G):
    """D (m×n), row e=(i,j): +1 at i, −1 at j (unweighted); w[e] = weight of edge e."""
    n = G.number_of_nodes()
    edges = sorted((min(u, v), max(u, v), d["weight"]) for u, v, d in G.edges(data=True))
    D = np.zeros((len(edges), n)); w = np.empty(len(edges))
    for e, (i, j, wij) in enumerate(edges):
        D[e, i], D[e, j], w[e] = 1.0, -1.0, wij
    return D, w

def soft_threshold(v, tau):
    """prox of tau·|.|, element-wise (tau may be a vector)."""
    return np.sign(v) * np.maximum(np.abs(v) - tau, 0.0)

def huber(r, delta):
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * r**2, delta * a - 0.5 * delta**2)

def prox_data(v, y, obs, rho, loss, delta):
    """argmin_z  sum_{i obs} loss(z_i − y_i)/rho + ½||z − v||²  (hidden nodes: z_i = v_i)."""
    z = v.copy()
    r = v[obs] - y[obs]
    if loss == "l2":
        s = rho * r / (1.0 + rho)
    elif loss == "huber":
        s = np.where(np.abs(r) <= delta * (1.0 + 1.0 / rho),
                     rho * r / (1.0 + rho),
                     r - np.sign(r) * delta / rho)
    else:
        raise ValueError(f"unknown loss: {loss}")
    z[obs] = y[obs] + s
    return z

_TV_CACHE = {}
def _tv_operators(G):
    """D, w and the Cholesky factor of I + DᵀD — computed once per graph."""
    hit = _TV_CACHE.get(id(G))
    if hit is None or hit[0] is not G:
        D, w = incidence_matrix(G)
        hit = (G, D, w, cho_factor(np.eye(D.shape[1]) + D.T @ D))
        _TV_CACHE[id(G)] = hit
    return hit[1:]

def tv_objective(x, G, y, mask=None, lam=0.1, loss="l2", delta=0.2):
    D, w, _ = _tv_operators(G)
    obs = np.ones(len(y), bool) if mask is None else np.asarray(mask, bool)
    r = x[obs] - y[obs]
    data = 0.5 * np.sum(r**2) if loss == "l2" else np.sum(huber(r, delta))
    return data + lam * np.sum(w * np.abs(D @ x))

def solve_tv(G, y, mask=None, lam=0.1, loss="l2", delta=0.2, rho=1.0, adaptive=True,
             max_iter=10000, eps_abs=1e-7, eps_rel=1e-5, return_info=False):
    """Graph TV recovery:  min Σ_obs loss(x_i − y_i) + lam·Σ w_ij|x_i − x_j|.

    mask     : bool array of length n, True = observed (None = all observed).
    lam      : model parameter (TV strength) — chosen with cv_score.
    loss     : "l2" (½r²) or "huber" (threshold delta, in the units of y).
    rho      : ADMM algorithm parameter (initial value) — changes speed only, not the solution.
    adaptive : adjust rho to balance the two residuals (Boyd et al. 2011, §3.4.1).
    """
    y = np.asarray(y, dtype=float); n = len(y)
    obs = np.ones(n, bool) if mask is None else np.asarray(mask, bool)
    D, w, chol = _tv_operators(G); m = len(w)
    z1 = np.where(obs, y, np.median(y[obs]))       # start hidden nodes at the median
    z2 = D @ z1
    u1, u2 = np.zeros(n), np.zeros(m)
    hist = []
    for k in range(1, max_iter + 1):
        x = cho_solve(chol, (z1 - u1) + D.T @ (z2 - u2))          # x-step
        Dx = D @ x
        z1_old, z2_old = z1, z2
        z1 = prox_data(x + u1, y, obs, rho, loss, delta)           # z1-step (data)
        z2 = soft_threshold(Dx + u2, lam * w / rho)                # z2-step (TV)
        u1 += x - z1; u2 += Dx - z2                                # dual step
        r_pri = np.sqrt(np.sum((x - z1)**2) + np.sum((Dx - z2)**2))
        s_dual = rho * np.linalg.norm((z1 - z1_old) + D.T @ (z2 - z2_old))
        eps_pri = np.sqrt(n + m) * eps_abs + eps_rel * max(np.sqrt(np.sum(x**2) + np.sum(Dx**2)),
                                                             np.sqrt(np.sum(z1**2) + np.sum(z2**2)))
        eps_dual = np.sqrt(n) * eps_abs + eps_rel * rho * np.linalg.norm(u1 + D.T @ u2)
        if return_info:
            hist.append((r_pri, s_dual, eps_pri, eps_dual,
                         tv_objective(x, G, y, obs, lam, loss, delta), rho))
        if r_pri < eps_pri and s_dual < eps_dual:
            break
        # Residual balancing: I + DᵀD has no rho, so changing rho needs no new factorisation;
        # only the scaled dual u = (dual)/rho is rescaled.
        if adaptive and k % 10 == 0:
            if r_pri > 10 * s_dual:
                rho *= 2.0; u1 /= 2.0; u2 /= 2.0
            elif s_dual > 10 * r_pri:
                rho /= 2.0; u1 *= 2.0; u2 *= 2.0
    if return_info:
        return x, {"iters": k, "converged": k < max_iter, "hist": np.array(hist)}
    return x

### Functions for the group: `solve_tv`

The solver is also saved as the module `src/method_5/method5_tv.py` (`from method5_tv import solve_tv`); the cell below checks that both give the same result.

**`solve_tv(G, y, mask=None, lam=0.1, loss="l2", delta=0.2, rho=1.0, adaptive=True, max_iter=10000, eps_abs=1e-7, eps_rel=1e-5, return_info=False)`**

| input | meaning |
|---|---|
| `G` | networkx graph, nodes `0 … n−1`, every edge has a `"weight"` |
| `y` | observed values, length `n` |
| `mask` | optional bool array, True = observed, False = hidden (its `y` is never used) |
| `lam` | TV strength λ (the group result uses the value chosen in Section 3) |
| `loss`, `delta` | `"l2"` (squared) or `"huber"` with threshold `delta`, in the units of `y` |
| `rho`, `adaptive` | ADMM step and residual balancing — speed only, keep the defaults |
| `max_iter`, `eps_abs`, `eps_rel` | stopping rule (Section 2.4) |
| `return_info` | True = also return `{"iters", "converged", "hist"}` (residuals, objective, ρ per iteration) |

Output: `x_hat`, a float array of length `n`, defined at every node (hidden nodes included).

**How others can use it**
- *As a smoother after method 6's flags:* `x = solve_tv(G, y, mask=~flag_outliers(G, y), lam=...)` — flagged nodes are treated as hidden, so TV refills them with the weighted median of their neighbours. Appendix C tests whether this helps.
- *In a median of several methods:* `results/method_5/xhat_5.npy` uses the node order 0..31 of `y`.

In [ ]:
# Example: solve once on the real data, then check the module gives the same result.
import sys
sys.path.insert(0, str(ROOT / "src" / "method_5"))
import method5_tv

x_demo, info_demo = solve_tv(G, y, lam=0.1, return_info=True)
print(f"lam = 0.1: {info_demo['iters']} ADMM iterations, converged = {info_demo['converged']}, "
      f"{len(np.unique(np.round(x_demo, 4)))} distinct values on {n} nodes")
assert np.allclose(x_demo, method5_tv.solve_tv(G, y, lam=0.1))
print("module method5_tv.solve_tv gives the same result: OK")

### 2.6 Correctness checks and convergence

The checks compare the solver with facts known independently of it:
1. **Building blocks:** $D^\top D$ equals the unweighted Laplacian; TV computed with $D$ equals TV computed edge by edge; soft-thresholding gives the textbook values.
2. **Optimality:** the problem is convex, so if no small random move lowers the objective, the solution is the global minimum. Done for both losses on a mask that hides about 20 % of the nodes.
3. **Limits:** $\lambda=0$ returns $y$; a huge $\lambda$ returns $\mathrm{mean}(y)$ (Section 2.2).
4. **Property 2 of Section 2.3:** each hidden node satisfies the weighted-median condition — the neighbour weight below it and above it differ by no more than the weight of neighbours tied with it.
5. **$\rho$ changes speed only:** for each $\lambda$ and starting $\rho_0$ the table prints the iteration count with fixed $\rho$ → adaptive $\rho$, and the largest distance from a high-precision reference solution. Fixed $\rho$ with a bad $\rho_0$ can hit the 10 000-iteration limit; adaptive $\rho$ removes that dependence.

In [ ]:
D, w = incidence_matrix(G)
L_unw = nx.laplacian_matrix(G, nodelist=range(n), weight=None).toarray()
assert np.allclose(D.T @ D, L_unw), "DᵀD must equal the unweighted Laplacian"
tv_loop = sum(d["weight"] * abs(y[u] - y[v]) for u, v, d in G.edges(data=True))
assert np.isclose(np.sum(w * np.abs(D @ y)), tv_loop), "TV via D differs from TV via a loop"
assert np.allclose(soft_threshold(np.array([-3, -.5, 0, .5, 3.]), 1), [-2, 0, 0, 0, 2])

# (1) Optimality: the problem is convex ⇒ no small perturbation may decrease the objective
rng = np.random.default_rng(123)
mask_test = rng.random(n) > 0.2
for loss in ["l2", "huber"]:
    p = dict(lam=0.3, loss=loss, delta=0.2)
    x_opt = solve_tv(G, y, mask=mask_test, **p)
    f0 = tv_objective(x_opt, G, y, mask_test, **p)
    worst = min(tv_objective(x_opt + 1e-3 * rng.normal(size=n), G, y, mask_test, **p) - f0
                for _ in range(2000))
    print(f"[{loss:5s}] f* = {f0:.6f} | min Δf over 2000 random perturbations = {worst:+.2e} (must be ≥ 0)")
    assert worst > -1e-9

# (2) Limits: lam→0 returns y; very large lam ⇒ constant signal = mean(y) (squared loss)
assert np.allclose(solve_tv(G, y, lam=0.0), y, atol=1e-5)
assert np.allclose(solve_tv(G, y, lam=1e3), np.mean(y), atol=1e-3)

# (3) Property 2: a hidden node is a weighted median of its neighbours.
#     0 ∈ subgradient of Σ_j w_ij|x_i − x_j|  ⇔  |weight below − weight above| ≤ weight tied
x_h = solve_tv(G, y, mask=mask_test, lam=0.3, eps_abs=1e-12, eps_rel=1e-11)
gap = 0.0
for i in np.where(~mask_test)[0]:
    nb = list(G[i]); wn = np.array([G[i][j]["weight"] for j in nb]); xn = x_h[nb]
    below = wn[xn < x_h[i] - 1e-6].sum(); above = wn[xn > x_h[i] + 1e-6].sum()
    gap = max(gap, abs(below - above) - (wn.sum() - below - above))
print(f"\nWeighted-median condition at {np.sum(~mask_test)} hidden nodes: worst violation = {max(gap, 0):.1e} (must be 0)")
assert gap <= 1e-6

# (4) rho changes speed only, not the solution; adaptive rho removes the dependence on the initial rho
RHOS = [0.01, 0.1, 1.0, 10.0, 100.0]
print("\nIterations per initial rho (fixed → adaptive), and largest deviation from the exact solution x*")
print("   λ   | " + " | ".join(f"rho0={r:<6g}" for r in RHOS) + " | max|x − x*|")
for lam_t in [0.01, 0.1, 1.0]:
    x_star = solve_tv(G, y, lam=lam_t, eps_abs=1e-11, eps_rel=1e-10)
    cells_, devs = [], []
    for rho in RHOS:
        x_f, i_f = solve_tv(G, y, lam=lam_t, rho=rho, adaptive=False, return_info=True)
        x_a, i_a = solve_tv(G, y, lam=lam_t, rho=rho, adaptive=True, return_info=True)
        cells_.append(f"{i_f['iters']:5d}→{i_a['iters']:<4d}")
        devs += [np.max(np.abs(x_f - x_star)), np.max(np.abs(x_a - x_star))]
    print(f" {lam_t:5.2f} | " + " | ".join(cells_) + f" | {max(devs):.1e}")
print("\nAll checks passed.")

In [ ]:
# Figure 1 — convergence on synthetic seed 0
x_true0, y_syn0 = make_synthetic(G, y, 0)
x0, info0 = solve_tv(G, y_syn0, lam=0.1, return_info=True)
H = info0["hist"]; it = np.arange(1, len(H) + 1)
f_star = solve_tv(G, y_syn0, lam=0.1, eps_abs=1e-12, eps_rel=1e-10)
f_star = tv_objective(f_star, G, y_syn0, lam=0.1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
ax.semilogy(it, H[:, 0], color=SERIES[0], label="primal residual $r^k$")
ax.semilogy(it, H[:, 1], color=SERIES[1], label="dual residual $s^k$")
ax.semilogy(it, H[:, 2], color=SERIES[0], lw=1, ls="--", alpha=.7, label=r"tolerance $\varepsilon_{pri}$")
ax.semilogy(it, H[:, 3], color=SERIES[1], lw=1, ls="--", alpha=.7, label=r"tolerance $\varepsilon_{dual}$")
ax.set(title="ADMM residuals", xlabel="iteration k"); ax.legend(fontsize=9)
ax = axes[1]
ax.semilogy(it, np.maximum(H[:, 4] - f_star, 1e-16), color=SERIES[0])
ax.set(title=r"Gap to the optimum $f(x^k)-f^\star$", xlabel="iteration k")
fig.suptitle(f"ADMM convergence — synthetic seed 0, λ = 0.1, ρ₀ = 1, adaptive ρ ({info0['iters']} iterations)", color=INK)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig5_convergence.png", dpi=160); plt.show()
print(f"Stopped after {info0['iters']} iterations; RMSE seed 0 = {rmse(x0, x_true0):.4f} (raw y: {rmse(y_syn0, x_true0):.4f})")

**Reading Figure 1.** Left: both residuals fall roughly on a straight line in log scale (linear convergence) until they drop below their dashed tolerances, where the solver stops; a kink is a $\rho$ update. Right: the gap between the objective and its optimum $f^\star$ falls the same way, so the stopping rule does not stop too early. The printed RMSE on seed 0 is a first comparison of TV with raw $y$.

## 3. Final result on the real data

**Parameters: chosen by `cv_score` on the real data, as in the group contract.**
- Grid: λ takes 25 log-spaced values in $[10^{-2.5}, 10^{0.5}]$; the loss is squared (`l2`) or Huber with $\delta\in\{0.05,0.1,0.2\}\times\mathrm{std}(y)$ (the synthetic noise has $\sigma=0.1\,\mathrm{std}(y)$). That makes 100 configurations.
- Rule: the lowest `cv_score`; ties go to `l2` (simpler model) and the smaller λ.
- $\rho_0=1$ with adaptive ρ (speed only, Section 2.6).
- Unlike method 6 (whose `k = 2.5` comes from the robust-statistics literature), TV has no standard value for λ: it depends on the scale of `y` and on the graph's weights. So λ **must** be chosen from the data, and is recomputed automatically when the MSSV changes.
- Appendix A shows the whole grid, compares it with synthetic RMSE, and explains why Huber is not selected.

In [ ]:
LAMS = np.logspace(-2.5, 0.5, 25)
s_y = np.std(y)
CONFIGS = [("l2", dict(loss="l2"))] + [
    (f"huber δ={c:g}·std(y)", dict(loss="huber", delta=c * s_y)) for c in (0.05, 0.1, 0.2)]
order = {name: i for i, (name, _) in enumerate(CONFIGS)}

t0 = time.time()
cv_grid = [(name, dict(base, lam=float(lam)), lam, cv_score(solve_tv, y, dict(base, lam=float(lam))))
           for name, base in CONFIGS for lam in LAMS]          # (name, params, λ, real cv_score)
best = min(cv_grid, key=lambda g: (round(g[3], 10), order[g[0]], g[2]))
BEST_NAME, BEST = best[0], best[1]
print(f"{len(cv_grid)} configurations × 5 folds in {time.time() - t0:.1f}s")
print(f"Chosen: {BEST_NAME}, λ = {BEST['lam']:.4g} → real cv_score = {best[3]:.4f}")

x_hat, info_real = solve_tv(G, y, return_info=True, **BEST)
assert x_hat.shape == y.shape and np.all(np.isfinite(x_hat))
np.save(RES_DIR / "xhat_5.npy", x_hat)
print(f"Real x_hat: {info_real['iters']} ADMM iterations, {len(np.unique(np.round(x_hat, 4)))} distinct values"
      f" on {n} nodes → saved results/method_5/xhat_5.npy")

### Figure: `y` versus `x_hat` on the graph

**Property 1 of Section 2.3, used to read the figure.** The optimality condition at an observed node $i$ (squared loss) is $y_i-\hat x_i=\lambda\sum_j w_{ij}g_{ij}$ with $g_{ij}\in\partial|\hat x_i-\hat x_j|\subset[-1,1]$, hence
$$|y_i-\hat x_i|\le\lambda d_i,\qquad d_i=\textstyle\sum_j w_{ij}\ \text{(weighted degree)}.$$
TV corrects each node by at most $\lambda d_i$. Nodes that **hit the cap** (black rings) are local extremes pulled as far as TV allows towards their neighbours. They are outlier candidates, but **not** a reliable outlier detector (on synthetic data this rule catches only about 1/3 of the true outliers).

In [ ]:
# Figure 3 — y, x_hat (same colour scale) and the correction y − x_hat; black ring = node at the cap |ψ(y_i − x̂_i)| = λ·d_i
resid = y - x_hat
deg_w = np.array([sum(d["weight"] for d in G[i].values()) for i in range(n)])
psi = resid if BEST["loss"] == "l2" else np.clip(resid, -BEST["delta"], BEST["delta"])
ratio = np.abs(psi) / (BEST["lam"] * deg_w)
assert ratio.max() <= 1 + 1e-2, "bound |ψ| ≤ λ·d_i violated — check the solution"
suspect = np.where(ratio > 0.99)[0]
norm = Normalize(vmin=min(y.min(), x_hat.min()), vmax=max(y.max(), x_hat.max()))
amax = np.abs(resid).max()
norm_d = Normalize(vmin=-amax, vmax=amax)
DIV = LinearSegmentedColormap.from_list("div_blue_red", ["#184f95", "#86b6ef", "#f0efec", "#f0a3a2", "#c23b3a"])
pos = positions

def draw_panel(ax, vals, cmap, nrm, title, ring=False):
    nx.draw_networkx_edges(G, pos, ax=ax, width=1.0, edge_color=AXIS)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=vals, cmap=cmap, vmin=nrm.vmin, vmax=nrm.vmax,
                           node_size=300, edgecolors=SURFACE, linewidths=2)
    if ring and len(suspect):
        nx.draw_networkx_nodes(G, pos, nodelist=list(suspect), ax=ax, node_color="none",
                               node_size=500, edgecolors=INK, linewidths=1.8)
    t = np.abs(nrm(vals) - 0.5) * 2 if cmap is DIV else nrm(vals)     # fill darkness, to pick the label colour
    on_dark = {i for i in G.nodes if t[i] > 0.55}
    nx.draw_networkx_labels(G, pos, {i: str(i) for i in G.nodes if i in on_dark}, ax=ax, font_size=7, font_color="#ffffff")
    nx.draw_networkx_labels(G, pos, {i: str(i) for i in G.nodes if i not in on_dark}, ax=ax, font_size=7, font_color=INK)
    ax.set_title(title); ax.axis("off")
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=nrm, cmap=cmap), ax=ax, orientation="horizontal",
                      shrink=0.75, pad=0.02, aspect=30)
    cb.outline.set_visible(False)
    return cb

fig, axes = plt.subplots(1, 3, figsize=(16, 6.2))
draw_panel(axes[0], y, SEQ, norm, "Observed y (noisy)").set_label("value", color=INK2)
draw_panel(axes[1], x_hat, SEQ, norm, f"Recovered x_hat (Graph TV, λ = {BEST['lam']:.3g})").set_label("value (same scale as y)", color=INK2)
draw_panel(axes[2], resid, DIV, norm_d,
           f"Correction made by TV: y − x_hat\nblack ring = at the cap |y − x_hat| = λ·d_i ({len(suspect)} nodes)",
           ring=True).set_label("y − x_hat (red: y higher, blue: y lower)", color=INK2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig5_graph_y_vs_xhat.png", dpi=160, bbox_inches="tight"); plt.show()

print(f"Largest correction max|y − x̂| = {np.abs(resid).max():.3f}; median = {np.median(np.abs(resid)):.3f}")
print("Nodes at the cap (outlier candidates):")
for i in suspect:
    print(f"  node {i:2d}: y = {y[i]:.3f}, x_hat = {x_hat[i]:.3f}, y − x_hat = {resid[i]:+.3f}, λ·d_i = {BEST['lam'] * deg_w[i]:.3f}")

**Reading Figure 3.** The left and middle panels share one colour scale, so a colour change between them is a correction. $\hat x$ shows patches of one colour: the flat regions. Right: red means $y$ was higher than $\hat x$ (TV pulled the node down), blue means it was pulled up. Black rings mark nodes whose correction reached the cap $\lambda d_i$: local extremes that TV pulled as far as it is allowed to. They are listed under the figure as outlier candidates. The `assert` in the cell also checks the bound $|y_i-\hat x_i|\le\lambda d_i$ at every node, a second independent check of the solution.

## 4. Result row for the group comparison table

One row per method, as agreed in the group doc: method, parameters, synthetic RMSE (mean ± std over seeds 0-19), `cv_score` on the real data. Two `cv_score` numbers are printed:
- `cv_score` with the contract's default split (`seed=0`) — the number used to choose λ;
- the mean over 5 splits (`cv_mean`), which is what **method 6 reports** in its row. Compare like with like (Appendix D, point 3).

In [ ]:
r_best = synth_rmse(BEST)
cv_best = cv_score(solve_tv, y, BEST)
cv_best_mean5 = cv_mean(solve_tv, y, BEST)

param_str = f"loss={BEST['loss']}" + (f", δ={BEST['delta']:.4g}" if BEST["loss"] == "huber" else "") + f", λ={BEST['lam']:.4g}, ρ₀=1 (adaptive)"
result = {
    "method": "5. Graph Total Variation (ADMM)",
    "representative_mssv": MSSV_DAI_DIEN,
    "best_params": {k: float(v) if not isinstance(v, str) else v for k, v in BEST.items()},
    "synthetic_rmse_mean": float(r_best.mean()), "synthetic_rmse_std": float(r_best.std()),
    "baseline_raw_y_rmse_mean": float(rmse_y.mean()),
    "cv_score_real": float(cv_best), "cv_score_real_mean5": cv_best_mean5,
    "admm_iterations_real": int(info_real["iters"]),
}
(RES_DIR / "result_5.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")

print("================ RESULT ROW — METHOD 5 ================")
print(f"Graph TV + ADMM | {param_str} | synthetic RMSE = {r_best.mean():.4f} ± {r_best.std():.4f}"
      f" | real cv_score = {cv_best:.4f} (mean of 5 splits: {cv_best_mean5:.4f})")
print("=======================================================")
print(f"Reference, return y unchanged: synthetic RMSE = {rmse_y.mean():.4f} ± {rmse_y.std():.4f}")
print(f"Seeds improved over y: {np.sum(r_best < rmse_y)}/20 → saved results/method_5/result_5.json")

**Reading the result row.**
- **Synthetic RMSE** — compare it with *return y unchanged*: any method has to beat that. "Seeds improved over y" says how consistent the gain is.
- **Real `cv_score`** — the median error when predicting hidden real nodes; the column that compares the 7 methods on the real data. It is compared with noisy `y`, so even a perfect method scores above 0.
- A more honest synthetic number, where λ is also chosen blind on each seed, is in Appendix B.3.

## 5. Write-up (Mục VI, 4 parts)

**1. Assumptions.** Observation = `x` + small independent noise + a *few* large outliers. The true `x` is **piecewise constant** on the graph: neighbours joined by heavy edges usually have equal values, with a few jumps between regions, so the edge differences `x_i − x_j` are sparse. Edge weights measure how much two sensors should agree.

**2. Loss / objective.** $\min_x \sum_{i\in\mathcal O}\ell(x_i-y_i)+\lambda\sum_{(i,j)}w_{ij}|x_i-x_j|$ with $\ell$ squared or Huber (Section 2.2). The absolute value keeps sharp jumps and makes whole regions flat (unlike $x^\top Lx$). The problem is convex. Two properties: each node is corrected by at most $\lambda d_i$ (bounded influence of an outlier), and a hidden node is predicted by the weighted median of its neighbours (Section 2.3).

**3. Optimisation and parameters.** ADMM with two splits $z_1=x$, $z_2=Dx$ (Boyd et al., 2011): a linear solve with the fixed matrix $I+D^\top D$ (one Cholesky factorisation for the whole notebook), the closed-form prox of the loss, soft-thresholding on each edge, a dual update (Section 2.4). Stopping rule: primal and dual residuals below absolute/relative tolerances (Boyd §3.3). ρ is adapted by residual balancing (speed only; Section 2.6 shows the answer does not depend on it). λ and the loss are chosen by `cv_score` on the real data over a grid of 100 configurations (Section 3, Appendix A). Correctness checks: building blocks, optimality under random perturbations, the λ limits, the weighted-median property, the bound $|y_i-\hat x_i|\le\lambda d_i$, and linear convergence (Figure 1).

**4. Experiments and limits.** Report the result row (Section 4), the λ-selection figure (Appendix A), Figure 1 and the `y` vs `x_hat` figure (Section 3). Limits (numbers in Appendix B.4):
- (a) *Assumption mismatch with the synthetic data:* `make_synthetic` builds a **smooth** `x` from the 5 lowest-frequency eigenvectors; TV approximates smooth slopes with small steps ("staircasing"). Methods based on $x^\top Lx$ have a structural advantage on this test, and a spectral filter with K = 5 (method 2) matches the generator by construction.
- (b) *Outliers are only partly removed:* an outlier larger than its node's cap $\lambda d_i$ moves only part of the way. Huber with a small δ does better on synthetic data, but the real-data `cv_score` does not select it.
- (c) *`cv_score` is noisy:* 32 nodes, about 6 hidden per fold; the CV curve is jagged, the chosen λ has variance, and CV compares with noisy `y`. The blind protocol (Appendix B.3) measures this cost.
- (d) *Improvements:* add a small $\gamma\,x^\top Lx$ ("elastic" TV) to reduce staircasing; choose δ from a robust noise estimate; combine with method 6's outlier flags (tested in Appendix C).

**References**
- S. Boyd, N. Parikh, E. Chu, B. Peleato, J. Eckstein (2011). *Distributed Optimization and Statistical Learning via the Alternating Direction Method of Multipliers.* Foundations and Trends in Machine Learning 3(1). Ch. 3 (scaled form, stopping rule, residual balancing), Ch. 6 (lasso / generalised lasso).
- N. Parikh, S. Boyd (2014). *Proximal Algorithms.* Foundations and Trends in Optimization 1(3). Soft-thresholding as the prox of $\ell_1$.
- Y.-X. Wang, J. Sharpnack, A. Smola, R. Tibshirani (2016). *Trend Filtering on Graphs.* JMLR 17. The graph fused lasso.
- D. Shuman et al. (2013). *The Emerging Field of Signal Processing on Graphs.* IEEE Signal Processing Magazine 30(3).

---
# Appendices (optional reading)

The main result above does not need them to be read. They contain the evidence behind it:
- **Appendix A**: the whole λ × loss grid, real-data `cv_score` against synthetic RMSE.
- **Appendix B**: synthetic data with a known `x`: one seed, seeds 0-19, the blind protocol, and the numbers behind the limitations.
- **Appendix C**: does TV improve when method 6's outlier flags are used as a mask?
- **Appendix D**: points to agree on at the group meeting, and the order of the 10-minute talk.

Run them in order (they use variables from the sections above).

## Appendix A. Parameter choice with `cv_score` (real data)

Section 3 chose the configuration from the real-data `cv_score` alone. Here every configuration also gets a synthetic RMSE (seeds 0-19), as a cross-check: does the λ chosen by CV also give a low error against the known `x`?

In [ ]:
t0 = time.time()
grid = []                          # each entry: (name, params, λ, cv_real, syn_mean, syn_std)
for name, p, lam, cv in cv_grid:
    r = synth_rmse(p)
    grid.append((name, p, lam, cv, r.mean(), r.std()))
print(f"Synthetic RMSE for {len(grid)} configurations × 20 seeds in {time.time() - t0:.1f}s")

print("\nTop 5 by real-data cv_score:")
for g in sorted(grid, key=lambda g: g[3])[:5]:
    print(f"  {g[0]:22s} λ={g[2]:.4g}  cv={g[3]:.4f}  RMSE_syn={g[4]:.4f}±{g[5]:.4f}")
oracle = min(grid, key=lambda g: g[4])
print(f"\n(Reference only — 'oracle' by synthetic RMSE: {oracle[0]}, λ={oracle[2]:.4g},"
      f" RMSE={oracle[4]:.4f}±{oracle[5]:.4f}; not used for selection because the true x is unknown in practice)")

In [ ]:
# Figure 2 — real-data cv_score and synthetic RMSE versus λ (two small panels, same λ axis)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), sharex=True)
for ci, (name, _) in enumerate(CONFIGS):
    rows = [g for g in grid if g[0] == name]
    lam_ = [g[2] for g in rows]
    axes[0].plot(lam_, [g[3] for g in rows], color=SERIES[ci], marker="o", ms=3.5, label=name)
    axes[1].plot(lam_, [g[4] for g in rows], color=SERIES[ci], marker="o", ms=3.5, label=name)
axes[1].axhline(rmse_y.mean(), color=MUTED, ls="--", lw=1)
axes[1].annotate("no processing (return y)", (LAMS[0], rmse_y.mean()), xytext=(0, 4),
                 textcoords="offset points", color=INK2, fontsize=9)
for ax in axes:
    ax.axvline(BEST["lam"], color=MUTED, lw=1, ls=":")
    ax.set_xscale("log"); ax.set_xlabel("λ (log scale)")
axes[0].annotate(f"chosen λ = {BEST['lam']:.3g}", (BEST["lam"], best[3]), xytext=(8, -14),
                 textcoords="offset points", color=INK, fontsize=9)
axes[0].set(title="Real-data cv_score (median |error|, lower is better)", ylabel="cv_score")
axes[1].set(title="Synthetic RMSE, mean over seeds 0–19", ylabel="RMSE")
axes[1].set_ylim(0, min(1.2, axes[1].get_ylim()[1]))
axes[0].set_ylim(0.2, None)
axes[0].legend(fontsize=9, loc="upper left")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig5_lambda_selection.png", dpi=160); plt.show()

**How to read the grid and Figure 2** (check again after setting the real MSSV):
- Left: real-data `cv_score` against λ, one line per loss. Small λ barely smooths, so a hidden node is predicted from few, noisy neighbours; large λ flattens everything towards one constant; the best value lies in between. The dotted line marks the chosen λ.
- Right: synthetic RMSE on the same λ axis, with raw `y` as the dashed line. If its minimum lies near the dotted line, the two criteria agree — evidence that CV picks a sensible λ although it can only compare against noisy `y`.
- The top-5 list shows how close the competitors are: differences in the third decimal are within the noise of a 32-node CV.
- The "oracle" (best synthetic RMSE) needs the true `x`; it only shows what TV could reach if we knew the answer. If it is a Huber configuration that CV does not choose, that is limitation (b) of Section 5.

## Appendix B. Synthetic data (known `x`)

`make_synthetic` (shared code, Section 1) builds a smooth `x` on our graph, adds Gaussian noise and 3 outliers, so RMSE against the true `x` can be measured. This only reflects the generator's assumptions, not necessarily the real data.

### B.1 Sanity check on one seed
Does TV beat raw `y` on seed 0? Which planted outliers (largest `|y_syn − x_true|`) reach the correction cap $\lambda d_i$?

In [ ]:
x_true, y_syn = SYN[0]
x_s = solve_tv(G, y_syn, **BEST)
planted = np.argsort(-np.abs(y_syn - x_true))[:3]
print("planted outliers (largest |y_syn − x_true|):", planted)
print("their size |y_syn − x_true|               :", np.round(np.abs(y_syn - x_true)[planted], 3))
print("their correction cap λ·d_i                 :", np.round(BEST["lam"] * deg_w[planted], 3))
print("their correction |y_syn − x_hat|           :", np.round(np.abs(y_syn - x_s)[planted], 3))
print(f"RMSE of y itself : {rmse(y_syn, x_true):.4f}")
print(f"RMSE of TV       : {rmse(x_s, x_true):.4f}")

**Reading.** When an outlier's size is larger than its cap, its correction equals the cap: TV pulls it back as far as it is allowed to, but not all the way (limitation (b)).

### B.2 RMSE over seeds 0-19
Per-seed comparison with raw `y` at the chosen parameters (the mean ± std is the result row of Section 4).

In [ ]:
print("seed :", " ".join(f"{s:5d}" for s in SEEDS))
print("y    :", " ".join(f"{v:5.3f}" for v in rmse_y))
print("TV   :", " ".join(f"{v:5.3f}" for v in r_best))
print(f"\nbaseline (x_hat = y): {rmse_y.mean():.4f} ± {rmse_y.std():.4f}")
print(f"TV, chosen params   : {r_best.mean():.4f} ± {r_best.std():.4f}  (better than y on {np.sum(r_best < rmse_y)}/20 seeds)")

### B.3 Blind protocol
For each seed, (loss, λ) is chosen by `cv_score` *on that seed's own* `y_syn` (every second λ of the grid, to save time), without looking at the true `x`; only then is RMSE measured. It is higher than B.2 because the choice itself is noisy: the gap is the price of not knowing `x`, and the most honest estimate of how the method does on a new case. Added to `result_5.json`.

In [ ]:
t0 = time.time(); r_blind, picks = [], []
for xt, ys in SYN:
    cand = [(cv_score(solve_tv, ys, dict(base, lam=float(lam))), order[name], lam, name, dict(base, lam=float(lam)))
            for name, base in CONFIGS for lam in LAMS[::2]]
    c = min(cand, key=lambda t: (round(t[0], 10), t[1], t[2]))
    r_blind.append(rmse(solve_tv(G, ys, **c[4]), xt)); picks.append((c[3], c[2]))
r_blind = np.array(r_blind)
result.update(synthetic_rmse_blind_mean=float(r_blind.mean()), synthetic_rmse_blind_std=float(r_blind.std()))
(RES_DIR / "result_5.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Blind protocol done in {time.time() - t0:.1f}s")
print(f"Synthetic RMSE, blind protocol = {r_blind.mean():.4f} ± {r_blind.std():.4f}"
      f"  (chosen params: {r_best.mean():.4f}; raw y: {rmse_y.mean():.4f})")
print("Losses picked per seed:", {nm: sum(p[0] == nm for p in picks) for nm, _ in CONFIGS})

### B.4 Numbers behind the limitations of Section 5

In [ ]:
# Evidence for limitations (a)-(c) of Section 5
caps = BEST["lam"] * deg_w                                      # correction cap λ·d_i per node (Section 2.3)
clip = (lambda r: r) if BEST["loss"] == "l2" else (lambda r: np.clip(r, -BEST["delta"], BEST["delta"]))
out_sizes, hits = [], 0
for xt, ys in SYN:
    err = np.abs(ys - xt)
    true_out = set(np.argsort(err)[-3:])                        # the 3 injected outliers = the 3 largest errors
    out_sizes += list(err[list(true_out)])
    at_cap = set(np.where(np.abs(clip(ys - solve_tv(G, ys, **BEST))) >= 0.99 * caps)[0])
    hits += len(true_out & at_cap)
print(f"Correction cap λ·d_i on this graph:            {caps.min():.2f} – {caps.max():.2f}")
print(f"Size of the injected outliers (20 seeds):       {min(out_sizes):.2f} – {max(out_sizes):.2f}")
print(f"Injected outliers whose node reaches the cap:   {hits}/{3 * len(SYN)}")
print(f"Gain over raw y at the chosen parameters:       {(1 - r_best.mean() / rmse_y.mean()) * 100:.1f} %")
print(f"Cost of choosing parameters blind (RMSE):       {r_blind.mean() - r_best.mean():+.4f}")

**Reading.**
- (a) The gain over raw `y` is moderate because the synthetic `x` is smooth, not piecewise constant.
- (b) Compare the cap range with the outlier sizes: outliers larger than their node's cap are only partly corrected, and only some of them reach the cap at all. So the "hits the cap" rule is used to read the figure in Section 3, not as an outlier detector.
- (c) The blind cost is what choosing parameters from noisy data costs.

## Appendix C. Using method 6's outlier flags with TV

**Question.** TV only partly corrects large outliers (limitation (b)). Method 6 provides a detector, `flag_outliers`, and a refill, `refill_outliers` (`src/method_6/method6_outliers.py`). Does combining them with TV help? Three variants; in every one the flags are **recomputed inside each CV fold** with that fold's mask, as method 6's README asks (otherwise hidden nodes influence the flags and the score looks too good):

| variant | what it does |
|---|---|
| `tv` | plain TV (the main result) |
| `tv_flag` | flagged nodes treated as hidden: `solve_tv(G, y, mask=mask & ~flag_outliers(G, y, mask=mask))`; TV refills them with the weighted median of their neighbours |
| `tv_refill` | flagged and hidden nodes first refilled by method 6 (weighted average of neighbours), then TV on the result; refilled values are treated as data |

Method 6 itself (`solve_two_stage`) is printed as a reference. Two tests:
1. **Our graph:** each variant picks its λ by real-data `cv_score` (every second λ of the grid, squared loss), then its synthetic RMSE (seeds 0-19) is measured.
2. **All 100 public graphs** (like method 6's Appendix C): 5 synthetic seeds per graph (500 runs) and the real-data `cv_score` of each graph, at fixed λ ∈ {0.05, 0.1, 0.2}. This part does not depend on the MSSV.

Takes about a minute.

In [ ]:
sys.path.insert(0, str(ROOT / "src" / "method_6"))
from method6_outliers import flag_outliers, refill_outliers, solve_two_stage

def tv_plain(G, y, mask=None, **p):
    return solve_tv(G, y, mask=mask, **p)

def tv_flag(G, y, mask=None, **p):
    obs = np.ones(len(y), bool) if mask is None else np.asarray(mask, bool)
    return solve_tv(G, y, mask=obs & ~flag_outliers(G, y, mask=obs), **p)

def tv_refill(G, y, mask=None, **p):
    obs = np.ones(len(y), bool) if mask is None else np.asarray(mask, bool)
    flagged = flag_outliers(G, y, mask=obs)
    y_ref = refill_outliers(G, np.where(obs, y, 0.0), flagged | ~obs)
    return solve_tv(G, y_ref, mask=obs, **p)

def m6(G, y, mask=None, **p):
    return solve_two_stage(G, y, mask=mask)

VARIANTS = {"tv": tv_plain, "tv_flag": tv_flag, "tv_refill": tv_refill}

# ---- (1) our graph: λ by real cv_score, then synthetic RMSE ----
t0 = time.time()
print("Our graph — λ chosen by real-data cv_score for each variant:")
for name, f in VARIANTS.items():
    cvs = [(cv_score(f, y, dict(lam=float(l))), l) for l in LAMS[::2]]
    cv_v, lam_v = min(cvs)
    r = np.array([rmse(f(G, ys, lam=float(lam_v)), xt) for xt, ys in SYN])
    print(f"  {name:9s} λ = {lam_v:<7.3g} real cv_score = {cv_v:.4f}   synthetic RMSE = {r.mean():.4f} ± {r.std():.4f}")
r6 = np.array([rmse(m6(G, ys), xt) for xt, ys in SYN])
print(f"  {'method 6':9s} {'':11s} real cv_score = {cv_score(m6, y, {}):.4f}   synthetic RMSE = {r6.mean():.4f} ± {r6.std():.4f}")
flags_real = np.flatnonzero(flag_outliers(G, y))
print(f"\nMethod 6 flags on the real y: {flags_real}; TV cap nodes (Section 3): {suspect}; "
      f"in both: {np.intersect1d(flags_real, suspect)}")

# ---- (2) all 100 public graphs ----
def synthetic_on(Gc, yc, seed):          # identical to make_synthetic, but on graph Gc
    nc = len(yc); Lc = nx.laplacian_matrix(Gc, nodelist=range(nc), weight="weight").toarray()
    rng = np.random.default_rng(seed)
    lam_, U = np.linalg.eigh(Lc)
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam_[:5]))
    x = np.median(yc) + np.std(yc) * x / np.std(x)
    sigma = 0.1 * np.std(yc)
    ys = x + rng.normal(0, sigma, nc)
    idx = rng.choice(nc, 3, replace=False)
    ys[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, ys

def cv_on(Gc, solve, yc, params, k=5, seed=0):   # identical to cv_score, but on graph Gc
    nc = len(yc); folds = np.random.default_rng(seed).permutation(nc) % k; err = []
    for f in range(k):
        mask = folds != f
        err.append(np.abs(solve(Gc, yc, mask=mask, **params)[~mask] - yc[~mask]))
    return np.median(np.concatenate(err))

with open(ROOT / "public_cases.json", encoding="utf-8") as _f:
    all_cases = json.load(_f)["cases"]
LAM_C = (0.05, 0.1, 0.2)
keys = ["m6"] + [f"{v}@{l:g}" for v in VARIANTS for l in LAM_C]
syn_all = {k: [] for k in ["y"] + keys}; cv_all = {k: [] for k in keys}
for case in all_cases.values():
    Gc = nx.Graph()
    Gc.add_nodes_from(range(len(case["positions"])))
    for u, v, wt in case["edges"]:
        Gc.add_edge(int(u), int(v), weight=float(wt))
    yc = np.asarray(case["y"], dtype=float)
    for sd in range(5):
        xt, ys = synthetic_on(Gc, yc, sd)
        syn_all["y"].append(rmse(ys, xt)); syn_all["m6"].append(rmse(m6(Gc, ys), xt))
        for v, f in VARIANTS.items():
            for l in LAM_C:
                syn_all[f"{v}@{l:g}"].append(rmse(f(Gc, ys, lam=l), xt))
    cv_all["m6"].append(cv_on(Gc, m6, yc, {}))
    for v, f in VARIANTS.items():
        for l in LAM_C:
            cv_all[f"{v}@{l:g}"].append(cv_on(Gc, f, yc, dict(lam=l)))
syn_all = {k: np.array(v) for k, v in syn_all.items()}; cv_all = {k: np.array(v) for k, v in cv_all.items()}

print(f"\nAll 100 public graphs ({time.time() - t0:.0f}s in total):")
print(f"  {'variant':14s} synthetic RMSE (500 runs)   mean real cv_score (100 graphs)")
print(f"  {'raw y':14s} {syn_all['y'].mean():.4f}")
for k in keys:
    print(f"  {k:14s} {syn_all[k].mean():.4f}                      {cv_all[k].mean():.4f}")
print("\n  Share of graphs where the variant has a LOWER real cv_score than plain TV at the same λ:")
for v in ("tv_flag", "tv_refill"):
    print(f"  {v:10s}", "  ".join(f"λ={l:g}: {np.mean(cv_all[f'{v}@{l:g}'] < cv_all[f'tv@{l:g}']) * 100:3.0f} %" for l in LAM_C))

**Result and decision: the main result does not use method 6's flags.** (Re-read the printout after setting the real MSSV; the 100-graph part does not change.)
- **`tv_flag` (flags as a mask) is worse than plain TV** in both tests: higher synthetic RMSE and a higher real `cv_score` on most graphs. Three reasons:
  1. a masked node in TV is filled by the weighted **median** of its neighbours (Property 2), which is coarser than a weighted average when the signal is smooth;
  2. the detector also flags about one clean node per run (method 6, Section 2), and each false flag throws away a good measurement;
  3. TV already limits each outlier's influence to $\lambda d_i$ (Property 1), so there is less left for a detector to fix than for a method with a quadratic penalty like method 1 or 6's Stage 2.
- **`tv_refill` lowers synthetic RMSE** at small λ, but **the real-data `cv_score` gets worse on most graphs**, and on our graph CV does not prefer it. The contract chooses by real-data CV, and the synthetic data favour averaging (smooth `x`), so we do not adopt it. It is worth mentioning at the meeting as a possible combination of methods 5 and 6 (Appendix D).
- **Consistency check:** on the real data, compare method 6's flagged nodes with TV's cap nodes (printed above): two independent methods pointing at the same node make it a strong outlier suspect.

## Appendix D. Notes for the group meeting

**Row for the 7-method comparison table:** printed in Section 4 and stored in `results/method_5/result_5.json`; `x_hat` is in `results/method_5/xhat_5.npy`; figures are in `figures/method_5/`.

**Points to agree on:**
1. **Real MSSV list.** All numbers here use the placeholder list. Once it is fixed, everyone re-runs with the **same** representative MSSV, otherwise the comparison table is meaningless.
2. **The synthetic set favours method 2** (limitation (a)). Weight the real-data `cv_score` more than synthetic RMSE when comparing methods.
3. **How the two numbers of the row are computed.** Synthetic RMSE: (a) parameters chosen on the real data and used for all 20 seeds (method 5's row), or (b) each seed picks its own (blind protocol, Appendix B.3). `cv_score`: the contract's single split (`seed=0`, method 5's main number) or the mean of 5 splits (what method 6 reports; method 5 prints both). The group should pick one convention for each so the 7 rows are comparable.
4. **Environment:** `nx.laplacian_matrix` needs **scipy**.
5. **Median of the best 2–3 methods:** `xhat_5.npy` uses the same node order 0..31 as `y` and can be used directly.
6. **If method 5 is chosen:** Mục VI is Section 5 of this notebook. Mục VII takes `incidence_matrix`, `soft_threshold`, `prox_data`, `_tv_operators`, `solve_tv` (Section 2.5, or `method5_tv.py`) with `make_synthetic` and `cv_score`, then calls `save_submission(x_hat)`.

**Order for the 10-minute talk:** the idea and TV vs roughness (Vocabulary + 2.2, 1 min) → the two properties (2.3, 2 min) → ADMM and Figure 1 (2.4–2.6, 2 min) → choosing λ (Section 3 + Appendix A, 2 min) → result row and the `y` vs `x_hat` figure (Sections 3–4, 2 min) → limitations and the method 6 combination (Section 5, Appendix C, 1 min).